In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [7]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model("claude-sonnet-4-6")
standard_model = init_chat_model("groq:openai/gpt-oss-20b")


@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 20:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

In [8]:
from langchain.agents import create_agent

agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [9]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

Hey there! I did water the office plant this morning—just gave it a good drink at 9 AM. It looks happy and green now. If you need anything else (like a quick check on its soil moisture or a pot swap), just let me know!


In [10]:
print(response["messages"][-1].response_metadata["model_name"])

openai/gpt-oss-20b


In [11]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

### When to replace the pot

A pot can usually stay in place for **2–3 years** for most indoor plants, but you’ll need to replace it sooner if any of the following signs appear:

| Sign | What it means | What to do |
|------|---------------|------------|
| **Roots growing out of drainage holes** | The plant is root‑bound and needs more space | Repot into a pot 1–2 inches larger (or replace the pot entirely) |
| **Plant keeps toppling or feels top‑heavy** | The pot is too small or the soil has become compacted | Use a larger pot or add fresh, light potting mix |
| **Soil dries out extremely fast or stays soggy** | Drainage or pot size is off | Check that the pot has adequate drainage holes; if not, use a pot that does |
| **Pot cracks or is visibly worn** | The pot itself is deteriorating | Replace it to avoid damage to the plant |
| **Visible mold or rot around the base** | The pot is retaining too much moisture | Switch to a pot with better drainage, use a well‑draining mix, and avoid

In [12]:
print(response["messages"][-1].response_metadata["model_name"])

openai/gpt-oss-20b
